In [ ]:
from ultralytics import YOLO

If we use yolo the last dataset is not usefull, we need the coordinates of the letters.

In [ ]:
import os
import random
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm

valores = ['2', '3', '4', '5', '6', '7', '8', 'b', 'c', 'd', 'e', 'f', 'g', 'm', 'n', 'p', 'w', 'x', 'y']
CHAR_TO_ID = {c: i for i, c in enumerate(valores)}

def add_noise(draw, width, height):
    # Puntos de ruido
    for _ in range(random.randint(60, 100)):
        x = random.randint(0, width)
        y = random.randint(0, height)
        draw.point((x, y), fill=(random.randint(80, 180), random.randint(80, 180), random.randint(80, 180)))
    
    # Líneas de ruido curvas/rectas
    for _ in range(random.randint(2, 4)):
        start = (random.randint(0, width // 3), random.randint(0, height))
        end = (random.randint(width * 2 // 3, width), random.randint(0, height))
        draw.line([start, end], fill=(random.randint(60, 150), random.randint(60, 150), random.randint(60, 150)), width=random.randint(1, 2))

def generate_captcha_yolo(width, height, text):
    img = Image.new("RGB", (width, height), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    
    font_size = 42
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", font_size)
    except IOError:
        try:
            font = ImageFont.truetype("arial.ttf", font_size)
        except IOError:
            font = ImageFont.load_default()

    boxes = []
    margin = 15
    char_step = (width - 2 * margin) // len(text)

    for i, char in enumerate(text):
        bbox = draw.textbbox((0, 0), char, font=font)
        char_w = bbox[2] - bbox[0]
        char_h = bbox[3] - bbox[1]

        x = margin + i * char_step + random.randint(-4, 4)
        y = random.randint(10, max(12, height - char_h - 20))

        char_color = (random.randint(10, 80), random.randint(10, 80), random.randint(10, 80))
        draw.text((x, y), char, font=font, fill=char_color)

        x_min = x + bbox[0]
        y_min = y + bbox[1]
        x_max = x + bbox[2]
        y_max = y + bbox[3]

        # Normalizar para YOLO: [0, 1]
        w_norm = (x_max - x_min) / width
        h_norm = (y_max - y_min) / height
        x_center = (x_min + x_max) / (2.0 * width)
        y_center = (y_min + y_max) / (2.0 * height)

        cls_id = CHAR_TO_ID[char]
        boxes.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

    # Añadir ruido visual encima de las letras
    add_noise(draw, width, height)

    return img, boxes

def generar_dataset_yolo(width, height, letras, output_dir, cantidad, split="train"):
    img_dir = os.path.join(output_dir, "images", split)
    lbl_dir = os.path.join(output_dir, "labels", split)
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    for idx in tqdm(range(cantidad), desc=f"Generando {split}"):
        palabra = "".join(random.choice(letras) for _ in range(5))
        img, boxes = generate_captcha_yolo(width, height, palabra)

        file_id = f"{split}_{idx:05d}"
        img.save(os.path.join(img_dir, f"{file_id}.png"))

        with open(os.path.join(lbl_dir, f"{file_id}.txt"), "w") as f:
            f.write("\n".join(boxes))

# Ejecutar generación
generar_dataset_yolo(280, 90, valores, "./dataset/yolo", cantidad=5000, split="train")
generar_dataset_yolo(280, 90, valores, "./dataset/yolo", cantidad=600, split="val")

Generando val: 100%|██████████| 600/600 [00:01<00:00, 437.18it/s]


In [ ]:
import yaml

def crear_data_yaml(output_dir, valores):
    data_dict = {
        'path': os.path.abspath(output_dir),
        'train': 'images/train',
        'val': 'images/val',
        'nc': len(valores),
        'names': valores
    }
    yaml_path = os.path.join(output_dir, "data.yaml")
    with open(yaml_path, 'w') as f:
        yaml.dump(data_dict, f, default_flow_style=False)

In [ ]:
from ultralytics import YOLO
import os
def main():
    valores = ['2', '3', '4', '5', '6', '7', '8', 'b', 'c', 'd', 'e', 'f', 'g', 'm', 'n', 'p', 'w', 'x', 'y']
    CHAR_TO_ID = {c: i for i, c in enumerate(valores)}
    dataset_path = "./dataset/yolo"
    yaml_file = crear_data_yaml(dataset_path, valores)
    model = YOLO("yolov8n.pt")
    model.train(
        data="dataset/yolo/data.yaml",
        epochs=30,
        imgsz=288, 
        batch=32,
        device=0,
        workers=4,
        project="captcha_yolo",
        name="run_01"
    )

if __name__ == "__main__":
    main()

New https://pypi.org/project/ultralytics/8.4.184 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.61 🚀 Python-3.14.8 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce GTX 1660, 5761MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=288, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, na

In [4]:
import cv2
from ultralytics import YOLO

model_path = "/home/tuxenjoyer69/Escritorio/devlab/python/2026-09-05 ML captchas resolver/runs/detect/captcha_yolo/run_01-3/weights/best.pt"
model = YOLO(model_path)

image_path = "./dataset/yolo/images/val/val_00000.png"

# 3. Inferencia
results = model(image_path, imgsz=288)

for r in results:
    annotated_frame = r.plot()

    detections = []
    for box in r.boxes:
        cls_id = int(box.cls[0].item())
        label = model.names[cls_id]
        x1 = box.xyxy[0][0].item()
        detections.append((x1, label))

    # Ordenar por x1 y concatenar letras
    detections.sort(key=lambda x: x[0])
    texto_resuelto = "".join([d[1] for d in detections])
    print(f"Texto detectado: {texto_resuelto}")

    cv2.imshow(f"Captcha resuelto: {texto_resuelto}", annotated_frame)
    cv2.waitKey(0)

cv2.destroyAllWindows()


image 1/1 /home/tuxenjoyer69/Escritorio/devlab/python/2026-09-05 ML captchas resolver/dataset/yolo/images/val/val_00000.png: 96x288 1 4, 1 8, 1 d, 1 n, 1 y, 4.8ms
Speed: 0.3ms preprocess, 4.8ms inference, 1.2ms postprocess per image at shape (1, 3, 96, 288)
Texto detectado: 8ynd4


In [3]:
from pathlib import Path
from ultralytics import YOLO

# Rutas del modelo y del conjunto de validación
model_path = "/home/tuxenjoyer69/Escritorio/devlab/python/2026-09-05 ML captchas resolver/runs/detect/captcha_yolo/run_01-3/weights/best.pt"
val_images_dir = Path("./dataset/yolo/images/val")
val_labels_dir = Path("./dataset/yolo/labels/val")

model = YOLO(model_path)

def obtener_ground_truth(label_file, class_names):
    """Lee el .txt de YOLO, ordena por coordenada X y devuelve el texto real."""
    if not label_file.exists():
        return None
    
    char_boxes = []
    with open(label_file, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                x_center = float(parts[1])
                char_boxes.append((x_center, class_names[cls_id]))
                
    # Ordenar caracteres de izquierda a derecha por x_center
    char_boxes.sort(key=lambda item: item[0])
    return "".join(item[1] for item in char_boxes)

total_captchas = 0
correctos = 0
fallos = []

# Listar todas las imágenes de validación
image_extensions = ("*.png", "*.jpg", "*.jpeg")
image_files = sorted([p for ext in image_extensions for p in val_images_dir.glob(ext)])

print(f"Evaluando {len(image_files)} captchas en {val_images_dir}...\n")

for img_path in image_files:
    label_path = val_labels_dir / f"{img_path.stem}.txt"
    gt_text = obtener_ground_truth(label_path, model.names)
    
    if gt_text is None:
        continue

    # Inferencia (verbose=False para no saturar la terminal)
    results = model(str(img_path), imgsz=288, verbose=False)
    
    detections = []
    for box in results[0].boxes:
        cls_id = int(box.cls[0].item())
        label = model.names[cls_id]
        x1 = box.xyxy[0][0].item()
        detections.append((x1, label))
        
    # Reconstrucción del texto predicho ordenado por coordenada X
    detections.sort(key=lambda item: item[0])
    pred_text = "".join(item[1] for item in detections)
    
    total_captchas += 1
    if pred_text == gt_text:
        correctos += 1
    else:
        fallos.append((img_path.name, gt_text, pred_text))

accuracy = (correctos / total_captchas) * 100 if total_captchas > 0 else 0.0

print("=" * 45)
print(f"Total evaluados:       {total_captchas}")
print(f"Captchas correctos:    {correctos}")
print(f"Captchas fallados:     {len(fallos)}")
print(f"Accuracy estricto:     {accuracy:.2f}%")
print("=" * 45)

if fallos:
    print("\nEjemplos de errores detectados (máx. 10):")
    for nombre, real, pred in fallos[:10]:
        print(f"Archivo: {nombre} | Real: '{real}' vs Pred: '{pred}'")

Evaluando 600 captchas en dataset/yolo/images/val...

Total evaluados:       600
Captchas correctos:    597
Captchas fallados:     3
Accuracy estricto:     99.50%

Ejemplos de errores detectados (máx. 10):
Archivo: val_00223.png | Real: 'gd87e' vs Pred: 'gb87e'
Archivo: val_00479.png | Real: 'fwdd3' vs Pred: 'fwdbd3'
Archivo: val_00517.png | Real: 'xn6bb' vs Pred: 'xn6bbd'
